# 1. Initializations

## 1.1 General imports

In [ ]:
### general
import logging
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
from typing import cast

### data management
import pandas as pd
import numpy as np

### machine learning (scikit-learn)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

### graphical
import matplotlib.pyplot as plt
# for jupyter notebook management
%matplotlib inline
import seaborn as sns

## 1.2 Dataframe imports

In [ ]:
import smartcheck.dataframe_common as dfc
import smartcheck.dataframe_project_specific as dfps

## 1.3 Preprocessing & Modeling imports

In [ ]:
import smartcheck.preprocessing_project_specific as pps
import smartcheck.modeling_project_specific as mps

# 2. Loading and Preprocessing

In [ ]:
df_cpt_raw = dfc.load_dataset_from_config('velo_comptage_ml_ready_data', sep=',', index_col=0)

if df_cpt_raw is not None and isinstance(df_cpt_raw, pd.DataFrame):
    df_cpt = df_cpt_raw.copy()

In [ ]:
df_cpt.info()

## 2.1 Preprocessing pipelines

In [ ]:
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    "date_et_heure_de_comptage",
    "orientation_compteur",
    "latitude",
    "longitude",
    "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    # "weather_code_wmo_code",
    "elevation",
    "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
    ("add_datetime_features", pps.DatetimePeriodicsTransformer(timestamp_col="date_et_heure_de_comptage")),
])

df_preproc = pipe_preproc.fit_transform(df_cpt)
if df_preproc is not None and isinstance(df_preproc, pd.DataFrame):
    df = df_preproc.copy()

In [ ]:
# Verification des distributions après preprocessing
df.info()
display(df.select_dtypes(include=np.number).describe())
display(df.select_dtypes(include='object').describe())

#### Feature selection (retour d'expérience sur VIF d'autre notebook pour même cible de comparaison)

In [ ]:
col_to_drop1 = [
    'date_et_heure_de_comptage_day_of_year',
    'date_et_heure_de_comptage_year',
    'date_et_heure_de_comptage_month',
    'date_et_heure_de_comptage_day',
    'date_et_heure_de_comptage_day_of_week',
    'date_et_heure_de_comptage_hour'
]

In [ ]:
col_to_drop2 = [
    'latitude',
    'longitude',
    'date_et_heure_de_comptage_sin_month',
    'date_et_heure_de_comptage_sin_week',
]

In [ ]:
col_to_drop3 = [
    'date_et_heure_de_comptage_cos_month',
]

In [ ]:
df = cast(pd.DataFrame, df.drop(columns=col_to_drop1+col_to_drop2+col_to_drop3))
df_feat_sel = df.copy()

## 2.2 Column Transformers

#### Categorical and Numerical

In [ ]:
num_col = list(df.drop(columns='comptage_horaire').select_dtypes(include=np.number).columns)
cat_col = list(df.select_dtypes(include='object').columns)
# s_scaler = StandardScaler()
mm_scaler = MinMaxScaler()
ohe_enc = OneHotEncoder(
    # drop='first', # évites la multicolinéarité mais déclenche des warning (les catégories inconnues participent a renforcer la catégorie droppée...)
    handle_unknown='ignore'
)
tr_num_col = Pipeline(
    steps = [
        ('standardisation', mm_scaler)
    ]
)
tr_cat_col = Pipeline(
    steps = [
        ('encoder', ohe_enc)
    ]
)
tr_columns = ColumnTransformer(
    transformers=[ 
        ('num_col_transf', tr_num_col, num_col),
        ('cat_col_transf', tr_cat_col, cat_col)
    ],
    remainder='drop',
)

# 3. Regression modeling

In [ ]:
liste_compteurs = [
    ('Pont de Bercy', 'NE-SO'),
    ('Pont de Bercy', 'NE-SO'),
    ('135 avenue Daumesnil', 'SE-NO'),
    ("180 avenue d'Italie", 'N-S'),
    ('27 quai de la Tournelle', 'NO-SE'),
    ('27 quai de la Tournelle', 'SE-NO'),
]

## 3.1 Linear Regresion

In [ ]:
pipe_linear_regression = Pipeline(
    steps= [
        ('prep', tr_columns), 
        ('reg',LinearRegression())
    ]
)

#### SANS AR(1) MA(24)

In [ ]:
models_lr_simple_results = {}
# pas d'aggrégation, juste un regroupement par nom de site et orientation (utilisé ensuite dans la boucle for)
grouped = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])
for compteur_id, df_compteur in grouped:
    if compteur_id not in liste_compteurs:
        continue
    # tri chrono + pipeline prétraitement + split
    df_compteur = df_compteur.sort_values("date_et_heure_de_comptage_local")
    X_train, X_train_dates, X_test, X_test_dates, y_train, y_test = \
        dfps.train_test_split_time_aware(
            df_compteur,
            timestamp_cols=["date_et_heure_de_comptage_utc", "date_et_heure_de_comptage_local"],
            target_col="comptage_horaire"
        )
    # pipeline + fit
    pipe_generic = pipe_linear_regression.fit(X_train, y_train)
    # predictions
    y_test_pred = pipe_generic.predict(X_test)
    # stockage des resultats
    models_lr_simple_results[compteur_id] = {
        "pipe":pipe_generic,
        "X_train":X_train,
        "X_train_dates":X_train_dates,
        "X_test":X_test,
        "X_test_dates":X_test_dates,
        "y_train":y_train,
        "y_test":y_test,
        "y_test_pred":y_test_pred,
    }

#### AVEC AR(1) MA(24)

In [ ]:
models_lr_ar1ma24_results = {}

# Grouping by site + orientation
grouped = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])

for compteur_id, df_compteur in grouped:
    if compteur_id not in liste_compteurs:
        continue
    df_compteur = df_compteur.sort_values("date_et_heure_de_comptage_local")

    # Chronological split
    X_train, X_train_dates, X_test, X_test_dates, y_train, y_test = dfps.train_test_split_time_aware(
        df_compteur,
        timestamp_cols=[
            "date_et_heure_de_comptage_utc",
            "date_et_heure_de_comptage_local"
        ],
        target_col="comptage_horaire"
    )
    # Fit AR feature transformer on training set
    tr_ar_feats = pps.AutoregressiveFeaturesTransformer(nb_ar=1, nb_mm=1, roll_wind=24)
    X_train_ar, X_train_dates_ar, y_train_ar = tr_ar_feats.fit_transform(
        X_train, X_train_dates, y_train
    )
    # Train model
    pipe_generic = pipe_linear_regression.fit(X_train_ar, y_train_ar)

    # Transform test set without refitting
    X_test_ar, X_test_dates_ar, y_test_ar = tr_ar_feats.transform_with_known_y(
        X_test, X_test_dates, y_test
    )

    # Predict
    y_test_pred = pipe_generic.predict(X_test_ar)

    # Save results
    models_lr_ar1ma24_results[compteur_id] = {
        "pipe":pipe_generic,
        "X_train":X_train_ar,
        "X_train_dates":X_train_dates_ar,
        "X_test":X_test_ar,
        "X_test_dates":X_test_dates_ar,
        "y_train":y_train_ar,
        "y_test":y_test_ar,
        "y_test_pred":y_test_pred,
    }

# 4 Performance analysis and visualisation

#### Compteurs "Pont de Bercy", "135 avenue Daumesnil", "180 avenue d'Italie", "27 quai de la Tournelle"

In [ ]:
# # Afficher une modélisation de compteur spécifique

# liste_model_results = {
#     "Regression linéaire simple":models_lr_simple_results,
#     "Regression linéaire AR1 MM24":models_lr_ar1ma24_results,
# }
# for compteur, (model_name, model_results) in itertools.product(liste_compteurs, liste_model_results.items()):
#     print("\n", "*"*80, "\n", f"Modèle {model_name} :","\n", "*"*80, "\n")
#     model_metrics = mps.compute_metrics(
#         model_results[compteur]["y_test"],
#         model_results[compteur]["y_test_pred"]
#     )
#     print("Metriques du modèle:",model_metrics)

#     fig_pred = mps.plot_predictions(
#         str(compteur),
#         model_results[compteur]["X_test_dates"], 
#         model_results[compteur]["y_test"], 
#         model_results[compteur]["y_test_pred"], 
#         periode_limite=('2025-04-01', '2025-04-16')
#     )
#     plt.show()

#     fig1_res, fig2_res, model_res_coeff = mps.compute_residuals_plot(
#         str(compteur),
#         model_results[compteur]["X_test_dates"], 
#         model_results[compteur]["y_test"], 
#         model_results[compteur]["y_test_pred"], 
#         periode_limite=('2025-04-01', '2025-04-16')
#     )
#     print("Pente de la droite de régression des résidus dans le temps (dérive) :", model_res_coeff)
#     plt.show()

#     fig_interp_list = mps.interpret_model(
#         model_results[compteur]
#     )
#     plt.show()

## Optimisation GridSearchCV - Ridge

In [ ]:
colonnes_a_exclure = [
    'Unnamed: 0', 'identifiant_du_compteur', 'nom_du_site_de_comptage',
    'orientation_compteur', 'arrondissement',
    'vacances_scolaires', 'weather_code_wmo_code_category'
]

X_train_dict = {}
y_train_dict = {}

compteurs_utiles = ['Pont de Bercy', '135 avenue Daumesnil', "180 avenue d'Italie", '27 quai de la Tournelle']

for compteur in compteurs_utiles:
    data = cast(pd.DataFrame, df[df['nom_du_site_de_comptage'] == compteur].copy())

    # Sécurité : conversion de la date et création de variables utiles
    if 'date_et_heure_de_comptage' in data.columns:
        data['date_et_heure_de_comptage'] = pd.to_datetime(data['date_et_heure_de_comptage'], errors='coerce')
        data['hour'] = data['date_et_heure_de_comptage'].dt.hour
        data['dayofweek'] = data['date_et_heure_de_comptage'].dt.dayofweek
        data['month'] = data['date_et_heure_de_comptage'].dt.month
        data.drop(columns=['date_et_heure_de_comptage'], inplace=True)

    # Suppression des colonnes inutiles
    data = data.drop(columns=colonnes_a_exclure, errors='ignore')

    # Encodage des colonnes catégorielles
    data = pd.get_dummies(data, drop_first=True)

    # Sélection stricte des colonnes numériques uniquement
    data = data.select_dtypes(include=['float64', 'int64'])

    # Suppression des lignes incomplètes
    data = data.dropna()

    if data.shape[0] < 10:
        print(f"⚠️ Trop peu de données pour {compteur}, ignoré.")
        continue

    X = data.drop(columns='comptage_horaire')
    y = data['comptage_horaire']

    X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=42)

    X_train_dict[compteur] = X_train
    y_train_dict[compteur] = y_train

In [ ]:
# Liste des compteurs
compteurs = df['nom_du_site_de_comptage'].unique()

# Colonnes à exclure des features
colonnes_a_exclure = [
    'Unnamed: 0', 'identifiant_du_compteur', 'nom_du_site_de_comptage',
    'date_et_heure_de_comptage', 'orientation_compteur', 'arrondissement',
    'vacances_scolaires', 'weather_code_wmo_code_category'
]

# Dictionnaires pour stocker X_train / y_train par compteur
X_train_dict = {}
y_train_dict = {}

for compteur in compteurs:
    data = df[df['nom_du_site_de_comptage'] == compteur].copy()

    # Nettoyage et sélection des features numériques
    data_model = cast(pd.DataFrame, data).drop(columns=colonnes_a_exclure, errors='ignore')
    data_model = pd.get_dummies(data_model, drop_first=True)

    # Suppression des lignes avec NaN (si besoin)
    data_model.dropna(inplace=True)

    # Séparation X / y
    X = data_model.drop(columns='comptage_horaire')
    y = data_model['comptage_horaire']

    # Split train/test
    X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=42)

    # Stockage
    X_train_dict[compteur] = X_train
    y_train_dict[compteur] = y_train

In [ ]:
param_grid = {
    'alpha': [0.01, 0.1, 1, 10, 100],
    'fit_intercept': [True, False],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sag']
}

best_params_ridge = {}

compteurs_a_traiter = ['Pont de Bercy', '135 avenue Daumesnil', "180 avenue d'Italie", '27 quai de la Tournelle']

colonnes_datetime_a_exclure = [
    'date_et_heure_de_comptage_utc',
    'date_et_heure_de_comptage_local'
]

for compteur in compteurs_a_traiter:
    print(f"\n--- Compteur : {compteur} ---")
    X_train = X_train_dict.get(compteur)
    y_train = y_train_dict.get(compteur)

    if X_train is None or y_train is None:
        print(f"Données manquantes pour le compteur {compteur}, passage au suivant.")
        continue

    # Supprimer les colonnes datetime si elles sont présentes
    X_train = X_train.drop(columns=[col for col in colonnes_datetime_a_exclure if col in X_train.columns], errors='ignore')

    # Ne garder que les colonnes numériques (sécurité supplémentaire)
    X_train = X_train.select_dtypes(include=['float64', 'int64'])

    model = Ridge()
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid,
                               cv=5, n_jobs=-1, verbose=0, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)
    best_params_ridge[compteur] = grid_search.best_params_
    print(f"Meilleurs paramètres : {grid_search.best_params_}")
    print(f"Meilleur score (MSE négatif) : {-grid_search.best_score_}")

In [ ]:
print(f"\nDtypes de X_train pour {compteur} :")
X_train = cast(pd.DataFrame, X_train)
print(X_train.dtypes)

# Optionnel : forcer la conversion uniquement des colonnes numériques
X_train = X_train.select_dtypes(include=['float64', 'int64'])

## Optimisation GridSearchCV - Lasso

In [ ]:
param_grid = {
    'alpha': [0.01, 0.1, 1, 10, 100],
    'fit_intercept': [True, False],
    'selection': ['cyclic', 'random'],
    'max_iter': [1000, 5000, 10000]
}

best_params_lasso = {}

compteurs_a_traiter = ['Pont de Bercy', '135 avenue Daumesnil', "180 avenue d'Italie", '27 quai de la Tournelle']

colonnes_datetime_a_exclure = [
    'date_et_heure_de_comptage_utc',
    'date_et_heure_de_comptage_local'
]

for compteur in compteurs_a_traiter:
    print(f"\n--- Compteur : {compteur} ---")
    X_train = X_train_dict.get(compteur)
    y_train = y_train_dict.get(compteur)

    if X_train is None or y_train is None:
        print(f"Données manquantes pour le compteur {compteur}, passage au suivant.")
        continue

    # Supprimer les colonnes datetime si présentes
    X_train = X_train.drop(columns=[col for col in colonnes_datetime_a_exclure if col in X_train.columns], errors='ignore')

    # Ne garder que les colonnes numériques
    X_train = X_train.select_dtypes(include=['float64', 'int64'])

    model = Lasso()
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid,
                               cv=5, n_jobs=-1, verbose=0, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)
    best_params_lasso[compteur] = grid_search.best_params_
    print(f"Meilleurs paramètres : {grid_search.best_params_}")

## Optimisation GridSearchCV - ElasticNet

In [ ]:
param_grid = {
    'alpha': [0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.5, 0.9],
    'fit_intercept': [True, False],
    'max_iter': [1000, 5000]
}

best_params_elasticnet = {}

compteurs_a_traiter = ['Pont de Bercy', '135 avenue Daumesnil', "180 avenue d'Italie", '27 quai de la Tournelle']

colonnes_datetime_a_exclure = [
    'date_et_heure_de_comptage_utc',
    'date_et_heure_de_comptage_local'
]

for compteur in compteurs_a_traiter:
    print(f"\n--- Compteur : {compteur} ---")
    X_train = X_train_dict.get(compteur)
    y_train = y_train_dict.get(compteur)

    if X_train is None or y_train is None:
        print(f"Données manquantes pour le compteur {compteur}, passage au suivant.")
        continue

    # Supprimer les colonnes datetime si présentes
    X_train = X_train.drop(columns=[col for col in colonnes_datetime_a_exclure if col in X_train.columns], errors='ignore')

    # Ne garder que les colonnes numériques
    X_train = X_train.select_dtypes(include=['float64', 'int64'])

    model = ElasticNet()
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid,
                               cv=5, n_jobs=-1, verbose=0, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)
    best_params_elasticnet[compteur] = grid_search.best_params_
    print(f"Meilleurs paramètres : {grid_search.best_params_}")

## Optimisation GridSearchCV - Random Forest

In [ ]:
# Grille réduite pour que ça s'exécute rapidement
param_grid = {
    'n_estimators': [100],
    'max_depth': [10, None],
    'min_samples_split': [2],
    'min_samples_leaf': [1],
    'max_features': ['sqrt']
}

best_params_random_forest = {}

compteurs_a_traiter = ['Pont de Bercy', '135 avenue Daumesnil', "180 avenue d'Italie", '27 quai de la Tournelle']
colonnes_datetime_a_exclure = ['date_et_heure_de_comptage_utc', 'date_et_heure_de_comptage_local']

for compteur in compteurs_a_traiter:
    print(f"\n--- Compteur : {compteur} ---")
    X_train = X_train_dict.get(compteur)
    y_train = y_train_dict.get(compteur)

    if X_train is None or y_train is None:
        print(f"Données manquantes pour le compteur {compteur}, passage au suivant.")
        continue

    # Nettoyage
    X_train = X_train.drop(columns=[col for col in colonnes_datetime_a_exclure if col in X_train.columns], errors='ignore')
    X_train = X_train.select_dtypes(include=['float64', 'int64'])

    if X_train.empty or y_train.empty or X_train.isnull().values.any() or y_train.isnull().values.any():
        print(f"Données invalides pour {compteur}, passage au suivant.")
        continue

    # GridSearchCV sur l'entraînement uniquement
    model = RandomForestRegressor(random_state=42)
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid,
                               cv=3, n_jobs=-1, verbose=0, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)
    best_params_random_forest[compteur] = grid_search.best_params_
    print(f"✅ Meilleurs paramètres pour {compteur} : {grid_search.best_params_}")

In [ ]:

# Résultats globaux
resultats_comparaison = []

for nom_site, orientation in liste_compteurs:
    df_compteur = df[(df["nom_du_site_de_comptage"] == nom_site) & 
                     (df["orientation_compteur"] == orientation)].copy()

    if df_compteur.shape[0] < 100:
        continue

    features = ['temperature_2m_c', 'rain_mm', 'snowfall_cm', 'jour_ferie']
    target = 'comptage_horaire'

    X = df_compteur[features]
    y = df_compteur[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    modeles = {
        "LinearRegression": {
            "default": LinearRegression(),
            "grid": LinearRegression()  # pas de grid search utile ici
        },
        "RandomForestRegressor": {
            "default": RandomForestRegressor(random_state=42),
            "grid": GridSearchCV(RandomForestRegressor(random_state=42),
                                 {"n_estimators": [50, 100], "max_depth": [5, 10]}, cv=3)
        }
    }

    for nom_modele, versions in modeles.items():
        for version, model in versions.items():
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            resultats_comparaison.append({
                "site": nom_site,
                "orientation": orientation,
                "modele": nom_modele,
                "version": version,
                "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
                "MAE": mean_absolute_error(y_test, y_pred),
                "R2": r2_score(y_test, y_pred)
            })

# Convertir en DataFrame
df_scores = pd.DataFrame(resultats_comparaison)
display(df_scores)

In [ ]:
# Graphique comparatif
for site in df_scores["site"].unique():
    df_site = df_scores[df_scores["site"] == site]
    for modele in df_site["modele"].unique():
        df_m = df_site[df_site["modele"] == modele]
        plt.figure(figsize=(6, 4))
        sns.barplot(data=df_m, x="version", y="RMSE")
        plt.title(f"{modele} - {site}")
        plt.ylabel("RMSE")
        plt.show()